# Building A Personal Corpus

**Notebook:** `01-building-a-personal-corpus.ipynb`

This notebook is part of the Personal Knowledge Engine project.

In [24]:
# ==========================================================
# Notebook 01: Building a Personal Corpus
# ==========================================================

import os
import glob
import pandas as pd
import pdfplumber
from pathlib import Path

In [25]:
os.makedirs("data/raw/projects", exist_ok=True)

os.makedirs("data/raw/journals", exist_ok=True)

In [26]:
sample_project = """
# AI Resume Matcher

Today I worked on the FAISS retrieval pipeline.

Important concepts:
- Sentence-BERT
- FAISS
- Cross Encoder
- Semantic Search

Need to improve Cross Encoder ranking.
"""

with open("data/raw/projects/ai_resume_matcher.md", "w", encoding="utf-8") as file:

    file.write(sample_project)

In [27]:
sample_journal = """
# Daily Journal

Today I read about local AI and privacy-first search systems.

Interesting ideas:
- Local embeddings
- Personal knowledge search
- Hybrid retrieval
"""

with open("data/raw/journals/2026-06-14.md", "w", encoding="utf-8") as file:

    file.write(sample_journal)

In [28]:
def read_markdown(file_path):

    with open(file_path, "r", encoding="utf-8") as file:

        text = file.read()

    return text

In [29]:
markdown_text = read_markdown("data/raw/projects/ai_resume_matcher.md")

print(markdown_text)


# AI Resume Matcher

Today I worked on the FAISS retrieval pipeline.

Important concepts:
- Sentence-BERT
- FAISS
- Cross Encoder
- Semantic Search

Need to improve Cross Encoder ranking.



In [30]:
# !pip install pymupdf

In [31]:
def read_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:

                text += page_text + "\n"

    return text

In [32]:
print(type(pdf_text))
print(len(pdf_text))
print(pdf_text[:1000])  # Preview first 1000 characters

<class 'str'>
755914
Praise for Natural Language Processing with Transformers
Pretrained transformer language models have taken the NLP world by
storm, while libraries such as Transformers have made them much
easier to use. Who better to teach you how to leverage the latest
breakthroughs in NLP than the creators of said library? Natural
Language Processing with Transformers is a tour de force, reflecting the
deep subject matter expertise of its authors in both engineering and
research. It is the rare book that offers both substantial breadth and
depth of insight and deftly mixes research advances with real-world
applications in an accessible way. The book gives informed coverage of
the most important methods and applications in current NLP, from
multilingual to efficient models and from question answering to text
generation. Each chapter provides a nuanced overview grounded in rich
code examples that highlights best practices as well as practical
considerations and enables you to put r

In [33]:
def read_text(file_path):

    with open(file_path, "r", encoding="utf-8") as file:

        return file.read()

In [34]:
SUPPORTED_TYPES = {".md": read_markdown, ".txt": read_text, ".pdf": read_pdf}

In [35]:
def load_documents(root_folder):

    documents = []

    for extension, loader in SUPPORTED_TYPES.items():

        pattern = os.path.join(root_folder, "**", f"*{extension}")

        files = glob.glob(pattern, recursive=True)

        for file_path in files:

            try:

                text = loader(file_path)

                documents.append(
                    {
                        "source": os.path.basename(file_path),
                        "path": file_path,
                        "type": extension.replace(".", ""),
                        "text": text,
                    }
                )

            except Exception as e:

                print(f"Error loading {file_path}: {e}")

    return documents

In [36]:
import warnings

warnings.filterwarnings("ignore")

In [37]:
import logging

logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [38]:
documents = load_documents("data/raw")

len(documents)

3

In [39]:
documents[0]

{'source': '2026-06-14.md',
 'path': 'data/raw\\journals\\2026-06-14.md',
 'type': 'md',
 'text': '\n# Daily Journal\n\nToday I read about local AI and privacy-first search systems.\n\nInteresting ideas:\n- Local embeddings\n- Personal knowledge search\n- Hybrid retrieval\n'}

In [40]:
corpus_df = pd.DataFrame(documents)

corpus_df.head()

,source,path,type,text
0,2026-06-14.md,data/raw\journals\2026-06-14.md,md,\n# Daily Journal\n\nToday I read about local ...
1,ai_resume_matcher.md,data/raw\projects\ai_resume_matcher.md,md,\n# AI Resume Matcher\n\nToday I worked on the...
2,book.pdf,data/raw\books\book.pdf,pdf,Praise for Natural Language Processing with Tr...


In [41]:
def infer_category(path):

    if "projects" in path:

        return "project"

    elif "journals" in path:

        return "journal"

    elif "books" in path:

        return "book"

    else:

        return "general"

In [42]:
corpus_df["category"] = corpus_df["path"].apply(infer_category)

corpus_df.head()

,source,path,type,text,category
0,2026-06-14.md,data/raw\journals\2026-06-14.md,md,\n# Daily Journal\n\nToday I read about local ...,journal
1,ai_resume_matcher.md,data/raw\projects\ai_resume_matcher.md,md,\n# AI Resume Matcher\n\nToday I worked on the...,project
2,book.pdf,data/raw\books\book.pdf,pdf,Praise for Natural Language Processing with Tr...,book


In [43]:
import re


def extract_date(filename):

    match = re.search(r"\d{4}-\d{2}-\d{2}", filename)

    if match:

        return match.group()

    return None

In [44]:
corpus_df["created_date"] = corpus_df["source"].apply(extract_date)

corpus_df.head()

,source,path,type,text,category,created_date
0,2026-06-14.md,data/raw\journals\2026-06-14.md,md,\n# Daily Journal\n\nToday I read about local ...,journal,2026-06-14
1,ai_resume_matcher.md,data/raw\projects\ai_resume_matcher.md,md,\n# AI Resume Matcher\n\nToday I worked on the...,project,None
2,book.pdf,data/raw\books\book.pdf,pdf,Praise for Natural Language Processing with Tr...,book,None


In [47]:
corpus_df.to_csv("data/processed/personal_corpus.csv", index=False)

print("Corpus saved successfully.")

Corpus saved successfully.


In [48]:
print("Total Documents:", len(corpus_df))

print()

print(corpus_df["category"].value_counts())

Total Documents: 3

category
journal    1
project    1
book       1
Name: count, dtype: int64
